# PhysAR-IDS -- NB4g (v6): converged setting C, cyber-only baseline, and what the model relies on under attack

Follow-up to NB4f. It answers two questions that NB4f left open.

1. **Is the 0.619 of setting C converged?** In NB4f, 38% of the rows in setting C stopped at the 25-round limit. Here the white-box search runs for up to **60 rounds** (`MAX_ITERS`). A **cyber-only 0.5 sigma** white-box run (setting E: electrical readings fixed) is added, on the same rows and seeds, so the value of electrical control is measured, not inferred: C minus E, with a burst-clustered interval.
2. **Does the electrical channel give any immunity, and which channel does the model rely on under attack?** (Here "electrical" = the 33 electrical features, "cyber" = the 60 network features.) Two measurements on the attacked rows:
   * **Attribution.** TreeSHAP values of the true-class margin on clean and on attacked rows: the share of total |SHAP| carried by the electrical group, and how the shift of the (benign minus true-class) margin splits between the two groups. TreeSHAP is additive, so the split is exact. Positive = toward benign.
   * **Counterfactual.** Undo the electrical part of the attack (restore the 33 electrical features) or the cyber part (restore the 60 cyber features) and re-predict: how many evaded rows stay evaded?

Everything else (threat model, power balance, frozen temporal features, evaluation rows, seeds) is as in NB4f. Needs the same input files. Runtime is dominated by setting C (up to 60 rounds); expect about 1.5 to 2 hours for five seeds on the GPU. The run resumes per seed and per setting.

In [ ]:
import os, sys, time, json, hashlib, warnings
import numpy as np, pandas as pd, joblib
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def find_project_dir(env_names=('PHYSAR_DIR',), marker='phase1_data_v6.pkl'):
    """Locate the folder that holds the pipeline files, wherever the notebook is run from.
    Order: environment variable(s) -> the current folder and its parents (and a SmartGrid_Dissertation folder next to or inside each) -> the usual places under the home folder."""
    from pathlib import Path
    cands = []
    for n in env_names:
        if os.environ.get(n): cands.append(Path(os.environ[n]).expanduser())
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents][:6]:
        cands += [p, p / 'SmartGrid_Dissertation'] + [p / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    home = Path.home()
    for extra in ('/content', '/content/drive/MyDrive', '/content/drive/MyDrive/smartgrid1', '/content/drive/MyDrive/smartgrid1/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1', '/content/drive/MyDrive/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1/SmartGrid_Dissertation', '/content/drive/My Drive', '/content/drive/My Drive/smartgrid1', '/content/drive/My Drive/smartgrid1/SmartGrid_Dissertation', '/content/drive/My Drive/smartrid1/SmartGrid_Dissertation', '/kaggle/working'):
        cands += [Path(extra), Path(extra) / 'SmartGrid_Dissertation']
    for base in ('Desktop', 'Documents', 'Downloads', 'OneDrive', 'OneDrive/Desktop', 'OneDrive/Documents', ''):
        b = home / base if base else home
        cands += [b / 'SmartGrid_Dissertation'] + [b / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    seen = set()
    for c in cands:
        try: c = c.resolve()
        except OSError: continue
        if c in seen: continue
        seen.add(c)
        if (c / marker).is_file(): return str(c)
    for root in [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')] + [home / b for b in ('Desktop', 'Documents', 'Downloads', 'OneDrive')]:   # shallow search, depth <= 3
        if not root.is_dir(): continue
        for dp, dn, fn in os.walk(root):
            if marker in fn: return dp
            if Path(dp).relative_to(root).parts.__len__() >= 3: dn[:] = []
    raise FileNotFoundError(f"Could not find {marker}. Set the environment variable PHYSAR_DIR to the folder that holds the pipeline files "
                            f"(e.g. in a cell, before running: os.environ['PHYSAR_DIR'] = r'C:\\path\\to\\SmartGrid_Dissertation'), or upload the files to the same folder as the notebook (on Colab: /content, or mount Google Drive first).")

PROJECT_DIR = find_project_dir(('NB4F_DIR', 'PHYSAR_DIR')); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = bool(int(os.environ.get('NB4F_SMOKE', '0')))
K_CAND = 12            # nearest split thresholds tried in each direction per quantity (plus the two interval ends)
MAX_ITERS = 60         # greedy rounds (one move per row per round; NB4f used 25)
CHUNK = 20000          # rows per search block
BLOCK = 400_000        # candidate rows scored per model call
N_PER_CLASS = 2000     # attack rows per class evaluated (None = all attack rows)
N_BOOT = 2000
SETTINGS = {
    'B_phys5_balance':      dict(cyber=None, eps_rel=0.05, balance=True),
    'C_cyber0p5_phys5_bal': dict(cyber=0.5,  eps_rel=0.05, balance=True),
    'E_cyber0p5_only':      dict(cyber=0.5,  eps_rel=0.0,  balance=True),     # electrical readings fixed
}
N_SHAP = 3000          # attack rows used for the SHAP / counterfactual analysis (stratified, fixed)
SETTINGS_TO_RUN = list(SETTINGS)
if SMOKE_TEST: SEEDS = SEEDS[:1]; K_CAND = 4; MAX_ITERS = 6; N_PER_CLASS = 150; N_BOOT = 200

os.environ['PYTHONHASHSEED'] = '42'
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'xgboost {xgb.__version__} | search backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"} | smoke {SMOKE_TEST} | working directory: {os.getcwd()}')

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train64 = np.asarray(data['X_train_scaled'], dtype=np.float64); X_test64 = np.asarray(data['X_test_scaled'], dtype=np.float64)
X_train = X_train64.astype(np.float32); X_test = X_test64.astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
scaler = joblib.load(f'standard_scaler_{PIPELINE_VERSION}.pkl'); MU = np.asarray(scaler.mean_, dtype=np.float64); SC = np.asarray(scaler.scale_, dtype=np.float64)
assert len(MU) == N_FEAT

atk_te = y_test != 0; X_atk_test64 = X_test64[atk_te]; y_atk_test = y_test[atk_te]
FEAT_LO, FEAT_HI = X_train.min(0).astype(np.float64), X_train.max(0).astype(np.float64)
PB_NAMES = ['V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower', 'Theta']
PB = np.array([FEATURE_COLS.index(c) for c in PB_NAMES]); DER = np.array([FEATURE_COLS.index(c) for c in ['voltage_imbalance', 'current_imbalance', 'power_factor']])
N_PHYS = len(PB)

def raw_of(Xs): return Xs[:, PB] * SC[PB] + MU[PB]
def derive(raw):
    V, I = raw[:, 0:3], raw[:, 3:6]; Vm, Im = V.mean(1), I.mean(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vimb = np.abs(V - Vm[:, None]).max(1) / np.where(Vm == 0, np.nan, Vm); cimb = np.abs(I - Im[:, None]).max(1) / np.where(Im == 0, np.nan, Im)
        S = np.hypot(raw[:, 6], raw[:, 7]); pf = raw[:, 6] / np.where(S == 0, np.nan, S)
    return np.nan_to_num(np.stack([vimb, cimb, pf], 1))
def ratio_of(raw):
    VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1)
    with np.errstate(divide='ignore', invalid='ignore'): return np.hypot(raw[:, 6], raw[:, 7]) / np.where(VI == 0, np.nan, VI)
def write_back(x, raw): x[:, PB] = (raw - MU[PB]) / SC[PB]; x[:, DER] = (derive(raw) - MU[DER]) / SC[DER]

raw_train = raw_of(X_train64); RAW_LO, RAW_HI = raw_train.min(0), raw_train.max(0)
r_tr = ratio_of(raw_train); r_tr = r_tr[np.isfinite(r_tr)]; BAND_LO, BAND_HI = np.percentile(r_tr, [0.5, 99.5])
r_te = ratio_of(raw_of(X_test64)); r_te = r_te[np.isfinite(r_te)]
der_err = np.abs((derive(raw_of(X_test64)) - MU[DER]) / SC[DER] - X_test64[:, DER]).max()
print(f'Features {N_FEAT} | cyber {N_CYBER} | physical channel {N_PHYS} | attack rows in the test period {len(X_atk_test64):,}')
print(f'Power balance on the test rows: 1st {np.percentile(r_te, 1):.5f}, median {np.median(r_te):.5f}, 99th {np.percentile(r_te, 99):.5f} | genuine band [{BAND_LO:.5f}, {BAND_HI:.5f}] | max derived-feature recomputation error {der_err:.1e}')
if not SMOKE_TEST:
    assert abs(np.median(r_te) - 1.0) < 1e-3 and np.percentile(r_te, 1) > 0.998 and np.percentile(r_te, 99) < 1.002, 'power balance not reproduced -- scaler / column order problem'
    assert der_err < 1e-3, 'derived-feature recomputation does not reproduce the stored values'
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
rng0 = np.random.RandomState(0)
pick = (np.arange(len(y_atk_test)) if N_PER_CLASS is None else np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), min(N_PER_CLASS, int((y_atk_test == c).sum())), replace=False) for c in range(1, 6)])))
X_wb, y_wb, burst_wb = X_atk_test64[pick], y_atk_test[pick], test_burst[pick]
print(f'Evaluation rows: {len(pick):,} ({np.bincount(y_wb)[1:].tolist()} per class) in {len(np.unique(burst_wb))} of {len(np.unique(test_burst))} attack bursts')
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1) if len(nb2_tuned) > 1 else 0:.4f}')

In [ ]:
def get_thresholds(clf):
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}
class Margin:
    def __init__(self, clf): self.b = clf.get_booster()
    def __call__(self, X):
        out = []
        if len(X) == 0: return np.zeros((0, 6))
        for s in range(0, len(X), BLOCK):
            out.append(np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + BLOCK], dtype=np.float32), predict_type='margin', validate_features=False)))
        return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)        # > 0  <=>  benign is the arg-max class

def make_ctx(x0):
    raw0 = raw_of(x0); rho0 = ratio_of(raw0); return raw0, np.where(np.isfinite(rho0), rho0, 1.0)
def finalize(x, raw, rho0, balance):
    """x (m, 93) float64 modified in place from the (m, 9) forged raw measurements: optional power-balance rescaling of (P, Q), then write-back with recomputed derived features."""
    if balance:
        VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1); S = np.hypot(raw[:, 6], raw[:, 7])
        f = np.where(S > 0, rho0 * VI / np.where(S > 0, S, 1.0), 1.0); f = np.where(np.isfinite(f) & (f > 0), f, 1.0); raw[:, 6] *= f; raw[:, 7] *= f
    write_back(x, raw)
def phys_interval(raw0, eps):
    a, b = raw0 * (1 - eps), raw0 * (1 + eps); lo, hi = np.minimum(a, b), np.maximum(a, b)
    return np.maximum(lo, np.minimum(RAW_LO, raw0)), np.minimum(hi, np.maximum(RAW_HI, raw0))

def cand_z(xz, a, b, T, K):
    """candidate values (standardized) for one quantity: the K nearest thresholds on each side inside [a, b], plus the interval ends. Returns (row index, value)."""
    nT = len(T); idx = np.searchsorted(T, xz, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)].astype(np.float64); ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)).astype(np.float64); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xz; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xz; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V)

def one_step(margin, x, x0, raw0, rho0, g, st, TH, K):
    n = len(x); best_g = g.copy(); best_kd = np.full(n, -1, np.int8); best_j = np.zeros(n, np.int64); best_v = np.zeros(n); bufs = []; size = [0]; eps = st['eps_rel']
    lo_raw, hi_raw = phys_interval(raw0, eps)
    def flush():
        if not bufs: return
        R = np.concatenate([b[0] for b in bufs]); KD = np.concatenate([b[1] for b in bufs]); J = np.concatenate([b[2] for b in bufs]); VV = np.concatenate([b[3] for b in bufs]); bufs.clear(); size[0] = 0
        for s in range(0, len(R), BLOCK):
            r, kd, jj, v = R[s:s + BLOCK], KD[s:s + BLOCK], J[s:s + BLOCK], VV[s:s + BLOCK]; Xc = x[r].copy()
            m = np.flatnonzero(kd == 0)
            if len(m): Xc[m, jj[m]] = v[m]
            m = np.flatnonzero(kd == 1)
            if len(m):
                raw = raw_of(Xc[m]); raw[np.arange(len(m)), jj[m]] = v[m]; sub = Xc[m]; finalize(sub, raw, rho0[r[m]], st['balance']); Xc[m] = sub
            gg = gap_of(margin(Xc.astype(np.float32)))
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_kd[rr] = kd[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    def add(R, kd, j, V):
        if len(R) == 0: return
        bufs.append((R, np.full(len(R), kd, np.int8), np.full(len(R), j, np.int64), V)); size[0] += len(R)
        if size[0] >= BLOCK: flush()
    for jm in (range(N_PHYS) if eps > 0 else []):                                 # physical measurements (none when the electrical readings are fixed)
        col = int(PB[jm]); T = TH.get(col)
        if T is None: continue
        a = (lo_raw[:, jm] - MU[col]) / SC[col]; b = (hi_raw[:, jm] - MU[col]) / SC[col]
        R, Z = cand_z(x[:, col], a, b, T, K)
        add(R, 1, jm, Z * SC[col] + MU[col])
    if st['cyber'] is not None:
        B = st['cyber']
        for col in CYBER_IDX:
            T = TH.get(int(col))
            if T is None: continue
            o = x0[:, col]; a = np.maximum(o - B, np.minimum(FEAT_LO[col], o)); b = np.minimum(o + B, np.maximum(FEAT_HI[col], o))
            R, Z = cand_z(x[:, col], a, b, T, K); add(R, 0, int(col), Z)
    flush(); return best_g, best_kd, best_j, best_v

def apply_moves(x, ids, kd, jj, vv, rho0, st):
    m = np.flatnonzero(kd == 0)
    if len(m): x[ids[m], jj[m]] = vv[m]
    m = np.flatnonzero(kd == 1)
    if len(m):
        sub = x[ids[m]]; raw = raw_of(sub); raw[np.arange(len(m)), jj[m]] = vv[m]; finalize(sub, raw, rho0[ids[m]], st['balance']); x[ids[m]] = sub

def whitebox(margin, X0, st, TH, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    x = X0.copy(); raw0, rho0 = make_ctx(X0); g = gap_of(margin(x.astype(np.float32))); init_evaded = g > 0; active = ~init_evaded; stalled = np.zeros(len(x), bool); log = []
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time()
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, kd, jj, vv = one_step(margin, x[ids], X0[ids], raw0[ids], rho0[ids], g[ids], st, TH, K)
            imp = kd >= 0; ii = ids[imp]; apply_moves(x, ii, kd[imp], jj[imp], vv[imp], rho0, st)
            g[ii] = gap_of(margin(x[ii].astype(np.float32))) if len(ii) else g[ii]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); log.append((it + 1, len(act), time.time() - t0))
        if verbose: print(f'      round {it + 1:>2}: active rows {len(act):>6} | {time.time() - t0:5.1f}s | evaded so far {(g > 0).mean():.3f}', flush=True)
    return x, g, init_evaded, stalled, log
print('white-box attack defined')

## 3. Self-test (first seed): same checks as NB4f, plus the electrical readings stay untouched in setting E

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test64), 40, replace=False); X0 = X_atk_test64[rows]; raw0, rho0 = make_ctx(X0); g0 = gap_of(margin(X0.astype(np.float32)))
    for sname in ('B_phys5_balance',):
        st = SETTINGS[sname]; bg, bkd, bj, bv = one_step(margin, X0.copy(), X0, raw0, rho0, g0, st, TH, K=10 ** 6); lo_raw, hi_raw = phys_interval(raw0, st['eps_rel']); worst = 0.0
        wide_gap = 0.0
        for i in range(len(rows)):
            best = g0[i]; best_wide = g0[i]
            for jm in range(N_PHYS):
                col = int(PB[jm]); T = TH.get(col)
                if T is None: continue
                a, b = (lo_raw[i, jm] - MU[col]) / SC[col], (hi_raw[i, jm] - MU[col]) / SC[col]; xz = X0[i, col]; Tf = T.astype(np.float64); Tdn = np.nextafter(T, np.float32(-np.inf)).astype(np.float64)
                fam = np.concatenate([[b] if b > xz else [], [a] if a < xz else [], Tf[(Tf > xz) & (Tf <= b)], Tdn[(Tf <= xz) & (Tdn >= a)]])          # exactly the attack's move family
                wide = np.concatenate([fam, Tf[(Tf >= a) & (Tf <= b)], Tdn[(Tdn >= a) & (Tdn <= b)]])                                               # plus same-cell points
                for pts, is_fam in ((fam, True), (wide, False)):
                    for z in np.unique(pts):
                        xr = X0[i:i + 1].copy(); raw = raw_of(xr); raw[0, jm] = z * SC[col] + MU[col]; finalize(xr, raw, rho0[i:i + 1], st['balance']); v = gap_of(margin(xr.astype(np.float32)))[0]
                        if is_fam: best = max(best, v)
                        best_wide = max(best_wide, v)
            wide_gap = max(wide_gap, best_wide - bg[i]); worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, {sname}): {bg[i]} < {best}'
        print(f'  {sname}: one_step matches the independent enumeration (largest shortfall {max(worst, 0):.2e}); extra same-cell points would add at most {max(wide_gap, 0):.2f} margin (heuristic gap, evasion stays a lower bound)')
    sub = X_atk_test64[rng.choice(len(X_atk_test64), 300, replace=False)]
    for sname, st in SETTINGS.items():
        xa, ga, ie, stl, lg = whitebox(margin, sub, st, TH, K=K_CAND, max_iters=4, verbose=False); r0, p0 = make_ctx(sub); raw_f = raw_of(xa)
        rel = np.abs(raw_f[:, [0, 1, 2, 3, 4, 5, 8]] / np.where(r0[:, [0, 1, 2, 3, 4, 5, 8]] == 0, np.nan, r0[:, [0, 1, 2, 3, 4, 5, 8]]) - 1); rel = np.nan_to_num(rel)
        assert rel.max() <= st['eps_rel'] + 1e-9, 'relative budget violated'
        dev = np.abs(ratio_of(raw_f) / p0 - 1); assert np.nanmax(dev) < 1e-9, 'power balance not kept'
        free = np.setdiff1d(np.arange(N_FEAT), np.concatenate([PB, DER] + ([CYBER_IDX] if st['cyber'] is not None else []))); assert np.abs(xa[:, free] - sub[:, free]).max() == 0, 'a frozen feature changed'
        if st['eps_rel'] == 0: assert np.abs(xa[:, PB] - sub[:, PB]).max() == 0 and np.abs(xa[:, DER] - sub[:, DER]).max() == 0, 'electrical readings changed in setting E'
        if st['cyber'] is not None: assert np.abs(xa[:, CYBER_IDX] - sub[:, CYBER_IDX]).max() <= st['cyber'] + 1e-9, 'cyber budget violated'
        assert (ga >= gap_of(margin(sub.astype(np.float32))) - 1e-9).all(), 'margin gap decreased'
    print('  constraints hold: relative budget, power balance, frozen features, cyber budget, monotone margin')
print('self-test defined')

## 4. Per-seed run, with the reliance analysis (resumable)

In [ ]:
ELEC_IDX = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX)
rs = np.random.RandomState(1); sidx = np.sort(np.concatenate([rs.choice(np.flatnonzero(y_wb == c), min(N_SHAP // 5, int((y_wb == c).sum())), replace=False) for c in range(1, 6)]))
def contrib(booster, X):
    c = booster.predict(xgb.DMatrix(np.ascontiguousarray(X, dtype=np.float32)), pred_contribs=True); c = np.asarray(c)
    assert c.ndim == 3 and c.shape[1] == 6 and c.shape[2] == N_FEAT + 1, f'unexpected contribution shape {c.shape}'
    return c[:, :, :N_FEAT]
def reliance(cpu, X0, xa, y, ev):
    n = len(sidx); ar = np.arange(n); yt = y[sidx]; cl = contrib(cpu.get_booster(), X0[sidx]); ad = contrib(cpu.get_booster(), xa[sidx]); out = {}
    def tr(c): return np.abs(c[ar, yt])
    for tag, c in (('clean', cl), ('adv', ad)):
        a = tr(c); out[f'share_elec_{tag}'] = float((a[:, ELEC_IDX].sum(1) / a.sum(1)).mean()); out[f'share_elec_{tag}_by_class'] = np.array([(a[yt == k][:, ELEC_IDX].sum(1) / a[yt == k].sum(1)).mean() for k in range(1, 6)])
    dG = (ad[:, 0] - ad[ar, yt]) - (cl[:, 0] - cl[ar, yt]); e = ev[sidx]
    out['dG_elec_all'] = float(dG[:, ELEC_IDX].sum(1).mean()); out['dG_cyber_all'] = float(dG[:, CYBER_IDX].sum(1).mean())
    out['dG_elec_evaded'] = float(dG[e][:, ELEC_IDX].sum(1).mean()) if e.any() else float('nan'); out['dG_cyber_evaded'] = float(dG[e][:, CYBER_IDX].sum(1).mean()) if e.any() else float('nan')
    out['dG_elec_notevaded'] = float(dG[~e][:, ELEC_IDX].sum(1).mean()) if (~e).any() else float('nan'); out['dG_cyber_notevaded'] = float(dG[~e][:, CYBER_IDX].sum(1).mean()) if (~e).any() else float('nan')
    return out
def counterfactual(cpu, X0, xa):
    xe = xa.copy(); xe[:, ELEC_IDX] = X0[:, ELEC_IDX]; xc = xa.copy(); xc[:, CYBER_IDX] = X0[:, CYBER_IDX]
    return cpu.predict(xe.astype(np.float32)) == 0, cpu.predict(xc.astype(np.float32)) == 0       # (cyber part only, electrical part only)
def summarise(xa, X0, cpu, init_evaded, stalled, log, ga, y):
    pred = cpu.predict(xa.astype(np.float32)); ev = pred == 0; raw_f, raw_0 = raw_of(xa), raw_of(X0); rel = np.nan_to_num(np.abs(raw_f / np.where(raw_0 == 0, np.nan, raw_0) - 1)); r_f = ratio_of(raw_f); inside = (r_f >= BAND_LO) & (r_f <= BAND_HI)
    ev_cy, ev_el = counterfactual(cpu, X0, xa)
    d = {'evaded': ev.copy(), 'ev_cyber_part_only': ev_cy, 'ev_electrical_part_only': ev_el, 'per_class_evasion': np.array([ev[y == c].mean() for c in range(1, 6)]), 'overall_evasion': float(ev.mean()), 'clean_benign_rate': float(init_evaded.mean()),
         'per_class_cyber_part_only': np.array([ev_cy[y == c].mean() for c in range(1, 6)]), 'per_class_electrical_part_only': np.array([ev_el[y == c].mean() for c in range(1, 6)]),
         'evaded_rows_inside_balance_band': float(inside[ev].mean()) if ev.any() else float('nan'), 'median_rel_change_VI_pct': float(100 * np.median(rel[:, :6].max(1))),
         'rows_stalled': float((stalled & ~ev).mean()), 'rows_iter_limit': float((~stalled & ~ev).mean()), 'rounds': len(log), 'search_time_s': float(sum(l[2] for l in log)), 'search_vs_cpu_oracle_disagree': float(((ga > 0) != ev).mean())}
    d.update(reliance(cpu, X0, xa, y, ev)); return d
R = {}; INFO = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack2g_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((best_params, K_CAND, MAX_ITERS, seed, N_PER_CLASS, N_SHAP, len(pick), float(X_wb[:5].sum()), [(k, sorted(SETTINGS[k].items(), key=str)) for k in SETTINGS_TO_RUN], 'nb4g-v1'))
    R[seed] = {}; INFO[seed] = {}
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed], INFO[seed] = saved['R'], saved['info']
    todo = [s for s in SETTINGS_TO_RUN if s not in R[seed]]
    if not todo: print(f'seed {seed}: all settings found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}: {len(todo)} setting(s) to run', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    tmp = f'_tmp_xgb4g_seed{seed}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp)
    margin = Margin(clf); TH = get_thresholds(clf); INFO[seed] = {'clean_weighted_f1': float(f1_score(y_test, cpu.predict(X_test), average='weighted')), 'n_split_features': len(TH)}
    print(f'  clean weighted F1 {INFO[seed]["clean_weighted_f1"]:.4f} | features used by the trees {len(TH)}', flush=True)
    if seed == SEEDS[0] and not R[seed]: selftest(clf, margin, TH)
    for sname in todo:
        st = SETTINGS[sname]; ts = time.time(); print(f'  [{sname}] attacking {len(X_wb)} rows ...', flush=True)
        xa, ga, ie, stl, lg = whitebox(margin, X_wb, st, TH, verbose=(sname.startswith('C'))); R[seed][sname] = summarise(xa, X_wb, cpu, ie, stl, lg, ga, y_wb); r = R[seed][sname]
        print(f'  [{sname}] per-class evasion {np.round(r["per_class_evasion"], 3).tolist()} overall {r["overall_evasion"]:.3f} | cyber part only {r["ev_cyber_part_only"].mean():.3f}, electrical part only {r["ev_electrical_part_only"].mean():.3f} | stalled {r["rows_stalled"]:.3f} iter-limit {r["rows_iter_limit"]:.3f} | rounds {r["rounds"]} | {(time.time() - ts) / 60:.1f} min', flush=True)
        joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Results: evasion by setting (mean +/- SD over seeds). NB4f's 25-round value for C was 0.619

In [ ]:
atk_names = class_names[1:]
def have(k): return [s for s in SEEDS if k in R[s]]
def arr(k, f): return np.stack([np.asarray(R[s][k][f]) for s in have(k)])
rows = []
for sname in SETTINGS_TO_RUN:
    if not have(sname): continue
    a = arr(sname, 'per_class_evasion'); o = arr(sname, 'overall_evasion'); n = len(a); st = SETTINGS[sname]
    row = {'setting': sname, 'cyber (sigma)': st['cyber'] if st['cyber'] is not None else '-', 'physical (rel.)': f'{100 * st["eps_rel"]:.0f}%', 'seeds': n}
    row.update({c: f'{a[:, i].mean():.3f} +/- {(a[:, i].std(ddof=1) if n > 1 else 0):.3f}' for i, c in enumerate(atk_names)}); row['ALL'] = f'{o.mean():.3f} +/- {(o.std(ddof=1) if n > 1 else 0):.3f}'
    row['inside band'] = f'{np.nanmean(arr(sname, "evaded_rows_inside_balance_band")):.3f}'; row['median rel. change V,I (%)'] = f'{arr(sname, "median_rel_change_VI_pct").mean():.2f}'
    row['stalled'] = f'{arr(sname, "rows_stalled").mean():.3f}'; row['iter-limit'] = f'{arr(sname, "rows_iter_limit").mean():.3f}'; rows.append(row)
wb_table = pd.DataFrame(rows); pd.set_option('display.width', 320); print(wb_table.to_string(index=False))
cb = np.mean([np.nan_to_num(R[s][SETTINGS_TO_RUN[0]]['clean_benign_rate']) for s in SEEDS if SETTINGS_TO_RUN[0] in R[s]]); print(f'Share of the evaluated attack rows already classified benign before any change: {cb:.3f}')

## 6. Burst-clustered intervals: convergence, and the value of electrical control

Resampling unit: attack bursts (350 in the full test set), 2,000 draws, pooled over seeds, as in NB4f. `C minus E` is the extra evasion that the electrical channel adds to a cyber attacker, on the same rows and seeds.

In [ ]:
ub = np.unique(test_burst); bi_full = np.searchsorted(ub, test_burst); nb_ = len(ub); idx_b = np.random.RandomState(42).randint(0, nb_, size=(N_BOOT, nb_)); bi_sub = bi_full[pick]
def bstat(vals, bi):
    cnt = np.bincount(bi, minlength=nb_).astype(float); out = [np.bincount(bi, weights=v.astype(float), minlength=nb_)[idx_b].sum(1) / cnt[idx_b].sum(1) for v in vals]
    return float(np.mean([v.mean() for v in vals])), float(np.percentile(np.mean(out, 0), 2.5)), float(np.percentile(np.mean(out, 0), 97.5))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ci_rows = []
def add_ci(label, vals):
    m, lo, hi = bstat(vals, bi_sub); ci_rows.append({'quantity': label, 'estimate': round(m, 4), 'lo': round(lo, 4), 'hi': round(hi, 4)}); return m
full = [k for k in SETTINGS_TO_RUN if len(have(k)) == len(SEEDS)]
for k in full: add_ci(f'{k}: white-box evasion', [np.asarray(R[s][k]['evaded']) for s in SEEDS])
if 'C_cyber0p5_phys5_bal' in full and 'E_cyber0p5_only' in full:
    C_, E_ = 'C_cyber0p5_phys5_bal', 'E_cyber0p5_only'
    add_ci('C minus E: what electrical control adds to a cyber attacker', [np.asarray(R[s][C_]['evaded'], float) - np.asarray(R[s][E_]['evaded'], float) for s in SEEDS])
if 'C_cyber0p5_phys5_bal' in full:
    add_ci('C: evasion if only the cyber part of the attack is kept', [np.asarray(R[s]['C_cyber0p5_phys5_bal']['ev_cyber_part_only']) for s in SEEDS])
    add_ci('C: evasion if only the electrical part of the attack is kept', [np.asarray(R[s]['C_cyber0p5_phys5_bal']['ev_electrical_part_only']) for s in SEEDS])
    add_ci('C: cyber part only minus full attack', [np.asarray(R[s]['C_cyber0p5_phys5_bal']['ev_cyber_part_only'], float) - np.asarray(R[s]['C_cyber0p5_phys5_bal']['evaded'], float) for s in SEEDS])
ci_table = pd.DataFrame(ci_rows); print(ci_table.to_string(index=False))
paired = pd.DataFrame()
if 'C_cyber0p5_phys5_bal' in full and 'E_cyber0p5_only' in full:
    pr = []
    for cls in range(0, 6):
        m = np.ones(len(y_wb), bool) if cls == 0 else (y_wb == cls)
        a = np.array([np.asarray(R[s][C_]['evaded'])[m].mean() for s in SEEDS]); b = np.array([np.asarray(R[s][E_]['evaded'])[m].mean() for s in SEEDS]); dd = a - b
        pr.append({'class': 'ALL' if cls == 0 else class_names[cls], 'C (cyber+electrical)': a.mean(), 'E (cyber only)': b.mean(), 'diff': dd.mean(), 'p_raw': 1.0 if dd.std(ddof=1) == 0 and dd.mean() == 0 else float(stats.ttest_rel(a, b).pvalue)})
    paired = pd.DataFrame(pr); paired['p_holm'] = holm(paired.p_raw.values); print('\nPaired t-tests over seeds, C vs E (model variance only):'); print(paired.round(4).to_string(index=False))

## 7. What does the model rely on under attack?

*Share of |SHAP| carried by the electrical group* (true-class margin, mean over rows) on clean and on attacked rows. *Shift of the margin (benign minus true class)*, split by group: positive = pushes the row toward benign. Rows are the stratified SHAP subsample. *Counterfactual*: evasion if only the cyber part (or only the electrical part) of the final attack is kept. Read the counterfactual rates against the floor of rows that are already classified benign without any change (printed under section 5).

In [ ]:
def ms(vals): v = np.asarray(vals, float); return f'{np.nanmean(v):.3f} +/- {(np.nanstd(v, ddof=1) if len(v) > 1 else 0):.3f}'
rel_rows = []
for k in full:
    g = lambda f: [R[s][k][f] for s in SEEDS]
    rel_rows.append({'setting': k, 'elec share |SHAP| clean': ms(g('share_elec_clean')), 'elec share |SHAP| attacked': ms(g('share_elec_adv')),
                     'margin shift, electrical (all rows)': ms(g('dG_elec_all')), 'margin shift, cyber (all rows)': ms(g('dG_cyber_all')),
                     'electrical (evaded rows)': ms(g('dG_elec_evaded')), 'cyber (evaded rows)': ms(g('dG_cyber_evaded')),
                     'evasion, cyber part only': ms([x.mean() for x in g('ev_cyber_part_only')]), 'evasion, electrical part only': ms([x.mean() for x in g('ev_electrical_part_only')]), 'evasion, full': ms(g('overall_evasion'))})
reliance_table = pd.DataFrame(rel_rows); print(reliance_table.T.to_string(header=False))
print('\nElectrical share of |SHAP| by true class (clean -> attacked):')
cls_rows = []
for k in full:
    for i, c in enumerate(atk_names):
        cls_rows.append({'setting': k, 'class': c, 'clean': round(float(np.mean([R[s][k]['share_elec_clean_by_class'][i] for s in SEEDS])), 3), 'attacked': round(float(np.mean([R[s][k]['share_elec_adv_by_class'][i] for s in SEEDS])), 3),
                         'evasion full': round(float(np.mean([R[s][k]['per_class_evasion'][i] for s in SEEDS])), 3), 'cyber part only': round(float(np.mean([R[s][k]['per_class_cyber_part_only'][i] for s in SEEDS])), 3), 'electrical part only': round(float(np.mean([R[s][k]['per_class_electrical_part_only'][i] for s in SEEDS])), 3)})
class_table = pd.DataFrame(cls_rows); print(class_table.to_string(index=False))

## 8. Save

In [ ]:
tables = {'whitebox': wb_table, 'burst_ci': ci_table, 'reliance': reliance_table, 'reliance_by_class': class_table}
if len(paired): tables['paired_C_vs_E'] = paired
for k_, t_ in tables.items(): t_.to_csv(f'attack2g_{k_}_{PIPELINE_VERSION}.csv', index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'SETTINGS': SETTINGS, 'K_CAND': K_CAND, 'MAX_ITERS': MAX_ITERS, 'N_PER_CLASS': N_PER_CLASS, 'N_SHAP': N_SHAP, 'tables': tables, 'info': INFO,
             'per_seed': {s: {k: {kk: vv for kk, vv in v.items() if kk not in ('evaded', 'ev_cyber_part_only', 'ev_electrical_part_only')} for k, v in R[s].items()} for s in SEEDS}, 'n_bursts': nb_}, f'attack2g_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB4g', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'K_CAND': K_CAND, 'MAX_ITERS': MAX_ITERS, 'N_PER_CLASS': N_PER_CLASS, 'N_SHAP': N_SHAP, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4g_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack2g_results_{PIPELINE_VERSION}.pkl', f'attack2g_whitebox_{PIPELINE_VERSION}.csv', f'attack2g_burst_ci_{PIPELINE_VERSION}.csv', f'attack2g_reliance_{PIPELINE_VERSION}.csv', f'nb4g_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)